# Experiment: Post CS-101 Non-Editing Gap Rank

Objective: rank non-editing or lower-infrastructure lanes after `CS-101` became the strongest Asia base-editing benchmark.

Success criteria:
- keep `CS-101`, `VGB-Ex01`, and luspatercept plus low-dose thalidomide as benchmarks, not Nakafa candidates;
- identify which lane still has an affordability, safety, or integrated-assay gap;
- avoid treatment advice and keep the result usable for lab-quote planning.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class Lane:
    """Candidate or benchmark lane used for post-CS-101 triage."""

    name: str
    endpoint: int
    human: int
    safety: int
    access: int
    assay: int
    infrastructure: int
    label: str
    role: str


def weighted_score(lane: Lane) -> float:
    """Return the original candidate-score weighted value."""
    return (
        0.30 * lane.endpoint
        + 0.25 * lane.human
        + 0.20 * lane.safety
        + 0.15 * lane.access
        + 0.10 * lane.assay
    )


def post_cs101_gap_score(lane: Lane) -> float:
    """Score lanes for an affordability-first, non-editing lab gap."""
    benchmark_penalty = 1.00 if lane.label == "benchmark_only" else 0.00
    blocked_penalty = 1.25 if lane.label in {"blocked", "novelty_blocked"} else 0.00
    caution_penalty = 0.60 if lane.safety <= 1 else 0.00
    infrastructure_penalty = 0.25 * lane.infrastructure
    labels_with_gap = {
        "integrated_assay_gap",
        "affordability_gap",
        "safety_gap",
    }
    gap_bonus = 0.60 if lane.label in labels_with_gap else 0.00

    return round(
        weighted_score(lane)
        + gap_bonus
        - benchmark_penalty
        - blocked_penalty
        - caution_penalty
        - infrastructure_penalty,
        2,
    )

## Plan

Hypothesis: after `CS-101`, pure HbF-editing novelty is blocked. The best remaining Nakafa lane should be non-editing, affordable enough to quote, and measurable with HbF plus alpha-globin/autophagy readouts.

Metrics:
- original weighted score from `2026-04-27-candidate-scoring-v0.md`;
- penalty for being a benchmark, blocked lane, high-caution safety lane, or infrastructure-heavy lane;
- bonus only when the lane has a defensible affordability, safety, or integrated-assay gap.


In [ ]:
lanes = [
    Lane(
        "Hydroxyurea",
        4,
        4,
        3,
        5,
        4,
        0,
        "benchmark_only",
        "affordable comparator",
    ),
    Lane(
        "Sirolimus",
        4,
        3,
        2,
        3,
        4,
        0,
        "integrated_assay_gap",
        "autophagy comparator",
    ),
    Lane(
        "PF-06409577 / PRKAB1-AMPKB1 probe",
        4,
        2,
        2,
        2,
        4,
        0,
        "integrated_assay_gap",
        "first expansion assay probe",
    ),
    Lane(
        "T-BDMC-like curcuminoid analog",
        3,
        2,
        2,
        3,
        3,
        0,
        "affordability_gap",
        "identity-gated screen seed",
    ),
    Lane(
        "Hepcidin-ferroportin-TMPRSS6",
        3,
        3,
        3,
        2,
        3,
        1,
        "benchmark_only",
        "iron-axis comparator",
    ),
    Lane(
        "Mitapivat",
        4,
        5,
        2,
        1,
        3,
        1,
        "benchmark_only",
        "approved disease-modifying benchmark",
    ),
    Lane(
        "Luspatercept + low-dose thalidomide",
        4,
        3,
        1,
        1,
        3,
        1,
        "benchmark_only",
        "closest non-editing pharmacologic benchmark",
    ),
    Lane(
        "LSD1 / DNMT1 epigenetic HbF targets",
        5,
        2,
        1,
        2,
        5,
        0,
        "safety_gap",
        "assay-only high-caution target class",
    ),
    Lane(
        "CRBN / WIZ / IKZF degrader class",
        5,
        2,
        1,
        2,
        4,
        0,
        "benchmark_only",
        "high-caution HbF comparator",
    ),
    Lane(
        "CS-101 / VGB-Ex01 editing",
        5,
        5,
        1,
        0,
        2,
        4,
        "benchmark_only",
        "editing benchmark, not Nakafa candidate",
    ),
]

ranked = sorted(
    (
        {
            "lane": lane.name,
            "role": lane.role,
            "label": lane.label,
            "weighted": round(weighted_score(lane), 2),
            "post_cs101_gap": post_cs101_gap_score(lane),
        }
        for lane in lanes
    ),
    key=lambda row: row["post_cs101_gap"],
    reverse=True,
)

ranked[:6]

## Results

The score is not a biological proof. It is a triage pressure test after `CS-101`. The result should be read as a lab-planning decision, not as patient guidance.

Expected interpretation:
- hydroxyurea remains the affordable positive comparator;
- sirolimus and PF-06409577 remain the most useful dual-readout expansion probes;
- T-BDMC-like chemistry is still alive but identity-gated;
- editing, luspatercept plus thalidomide, mitapivat, hepcidin-axis drugs, and degrader classes are benchmarks unless a lab-specific question requires them.


In [ ]:
decision = {
    "first_expansion_probe": "PF-06409577 / PRKAB1-AMPKB1 probe",
    "required_comparator": "Hydroxyurea",
    "autophagy_comparator": "Sirolimus",
    "conditional_seed": "T-BDMC-like curcuminoid analog",
    "blocked_as_treatment_claim": [
        "CS-101 / VGB-Ex01 editing",
        "Luspatercept + low-dose thalidomide",
        "LSD1 / DNMT1 epigenetic HbF targets",
        "CRBN / WIZ / IKZF degrader class",
    ],
}

decision

## Next steps

- Move the durable conclusion into a finding note and the source registry.
- Keep the first lab quote panel unchanged.
- Add PF-06409577 only as an expansion probe when the lab can document identity and run HbF, alpha-globin/autophagy, maturation, viability, and hemolysis readouts.
